In [1]:
# ==================================================
# RECUPERAR CONEXIÓN CON SQL SERVER
# ==================================================

import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from dotenv import load_dotenv

# Carpeta principal del proyecto
RUTA_PROYECTO = Path.cwd()

# Cargar configuración del proyecto
load_dotenv(RUTA_PROYECTO / ".env")

# Localizar el archivo database.py
RUTA_SRC = RUTA_PROYECTO / "src"

if str(RUTA_SRC) not in sys.path:
    sys.path.insert(0, str(RUTA_SRC))

# Importar conexión original
from database import engine_sqlserver

# Conectar con SQL Server
engine = engine_sqlserver()

with engine.connect() as conexion:
    nombre_bd = conexion.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

print("Conexión establecida correctamente.")
print("Base de datos:", nombre_bd)


Conexión establecida correctamente.
Base de datos: RedSaludCarazoDW_V2



## PASO 2. Carga y verificación del dataset predictivo

### Objetivo
Extraer los registros de citas médicas desde SQL Server y comprobar que el conjunto de datos está preparado para las siguientes etapas de minería de datos.

### Actividades
- Cargar los registros de dbo.dm_dataset_prediccion.
- Identificar las columnas y sus tipos de datos.
- Verificar los valores faltantes.
- Comprobar la variable objetivo no_asistio.
- Verificar los identificadores y las fechas.
- Analizar la distribución mensual de las citas.
- Guardar evidencias en archivos CSV.

### Resultado esperado
Obtener un conjunto de datos verificable para su posterior preparación, división cronológica y entrenamiento de modelos predictivos.


In [3]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# MINERÍA DE DATOS
# PASO 2. CARGA Y VERIFICACIÓN DEL DATASET
# ============================================================

import pandas as pd
from pathlib import Path
from sqlalchemy import text
from IPython.display import display


# ============================================================
# 1. CARGAR DATASET PREDICTIVO
# ============================================================

consulta = text("""
    SELECT *
    FROM dbo.dm_dataset_prediccion
""")

with engine.connect() as conexion:

    datos_predictivos = pd.read_sql_query(
        consulta,
        conexion
    )

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PASO 2. VERIFICACIÓN DEL DATASET PREDICTIVO")
print("=" * 70)

print("\n1. DIMENSIONES DEL DATASET")
print("Registros:", len(datos_predictivos))
print("Columnas:", len(datos_predictivos.columns))

if datos_predictivos.empty:
    raise ValueError(
        "El dataset predictivo está vacío."
    )


# ============================================================
# 2. IDENTIFICAR COLUMNAS
# ============================================================

diccionario_columnas = pd.DataFrame({
    "Columna": datos_predictivos.columns,
    "Tipo_dato": [
        str(tipo)
        for tipo in datos_predictivos.dtypes
    ],
    "Valores_nulos": [
        int(datos_predictivos[campo].isna().sum())
        for campo in datos_predictivos.columns
    ],
    "Valores_distintos": [
        int(datos_predictivos[campo].nunique(
            dropna=True
        ))
        for campo in datos_predictivos.columns
    ]
})

print("\n2. DICCIONARIO DE COLUMNAS")
display(diccionario_columnas)


# ============================================================
# 3. REVISAR VALORES FALTANTES
# ============================================================

resumen_nulos = pd.DataFrame({
    "Columna": datos_predictivos.columns,
    "Nulos": datos_predictivos.isna().sum().values
})

resumen_nulos["Porcentaje_nulos"] = (
    resumen_nulos["Nulos"]
    / len(datos_predictivos)
    * 100
).round(2)

print("\n3. VALORES FALTANTES")
display(resumen_nulos)

print(
    "Total de valores nulos:",
    int(datos_predictivos.isna().sum().sum())
)


# ============================================================
# 4. VERIFICAR VARIABLE OBJETIVO
# ============================================================

objetivo = "no_asistio"

if objetivo not in datos_predictivos.columns:
    raise ValueError(
        "No se encontró la variable objetivo no_asistio."
    )

valores_objetivo = pd.to_numeric(
    datos_predictivos[objetivo],
    errors="coerce"
)

if not valores_objetivo.isin([0, 1]).all():
    raise ValueError(
        "La variable objetivo contiene valores "
        "nulos o diferentes de 0 y 1."
    )

distribucion_objetivo = (
    valores_objetivo
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("no_asistio")
    .reset_index(name="Cantidad")
)

distribucion_objetivo["Porcentaje"] = (
    distribucion_objetivo["Cantidad"]
    / len(datos_predictivos)
    * 100
).round(2)

distribucion_objetivo["Resultado"] = (
    distribucion_objetivo["no_asistio"]
    .map({
        0: "Asistió",
        1: "No asistió"
    })
)

print("\n4. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO")
display(
    distribucion_objetivo[
        [
            "no_asistio",
            "Resultado",
            "Cantidad",
            "Porcentaje"
        ]
    ]
)


# ============================================================
# 5. VERIFICAR IDENTIFICADORES
# ============================================================

if "id_cita" not in datos_predictivos.columns:
    raise ValueError(
        "No se encontró la columna id_cita."
    )

ids_duplicados = int(
    datos_predictivos["id_cita"].duplicated().sum()
)

ids_nulos = int(
    datos_predictivos["id_cita"].isna().sum()
)

print("\n5. VERIFICACIÓN DE IDENTIFICADORES")
print("Identificadores duplicados:", ids_duplicados)
print("Identificadores nulos:", ids_nulos)

if ids_duplicados > 0 or ids_nulos > 0:
    raise ValueError(
        "Se detectaron problemas en los identificadores."
    )


# ============================================================
# 6. VERIFICAR FECHAS
# ============================================================

if "fecha_cita" not in datos_predictivos.columns:
    raise ValueError(
        "No se encontró fecha_cita."
    )

fechas = pd.to_datetime(
    datos_predictivos["fecha_cita"],
    errors="coerce"
)

fechas_invalidas = int(fechas.isna().sum())

print("\n6. VERIFICACIÓN DE FECHAS")
print("Fechas inválidas:", fechas_invalidas)

if fechas_invalidas > 0:
    raise ValueError(
        "Se encontraron fechas inválidas."
    )

print("Fecha mínima:", fechas.min())
print("Fecha máxima:", fechas.max())


# ============================================================
# 7. DISTRIBUCIÓN MENSUAL
# ============================================================

tabla_mensual = (
    pd.DataFrame({
        "Periodo": fechas.dt.to_period("M"),
        "no_asistio": valores_objetivo.astype(int)
    })
    .groupby("Periodo")
    .agg(
        Citas=("no_asistio", "size"),
        Inasistencias=("no_asistio", "sum")
    )
    .reset_index()
)

tabla_mensual["Asistencias"] = (
    tabla_mensual["Citas"]
    - tabla_mensual["Inasistencias"]
)

tabla_mensual["Tasa_inasistencia_pct"] = (
    tabla_mensual["Inasistencias"]
    / tabla_mensual["Citas"]
    * 100
).round(2)

tabla_mensual["Periodo"] = (
    tabla_mensual["Periodo"].astype(str)
)

print("\n7. DISTRIBUCIÓN MENSUAL")
display(tabla_mensual)


# ============================================================
# 8. MOSTRAR REGISTROS DE EJEMPLO
# ============================================================

print("\n8. PRIMEROS CINCO REGISTROS")
display(datos_predictivos.head())



# ============================================================
# 9. GUARDAR EVIDENCIAS
# ============================================================

# Identificar la carpeta de trabajo
proyecto = Path.cwd()

# Definir carpeta donde guardaremos los CSV
carpeta_fuentes = (
    proyecto
    / "resultados"
    / "01_fuentes"
)

# Crear carpetas si todavía no existen
carpeta_fuentes.mkdir(
    parents=True,
    exist_ok=True
)

# Guardar diccionario de columnas
diccionario_columnas.to_csv(
    carpeta_fuentes / "diccionario_columnas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar revisión de valores faltantes
resumen_nulos.to_csv(
    carpeta_fuentes / "valores_faltantes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar distribución de asistencia
distribucion_objetivo.to_csv(
    carpeta_fuentes / "distribucion_objetivo.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar distribución mensual
tabla_mensual.to_csv(
    carpeta_fuentes / "distribucion_mensual.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nARCHIVOS GUARDADOS CORRECTAMENTE")
print("Ubicación:", carpeta_fuentes)

for archivo in carpeta_fuentes.glob("*.csv"):
    print("-", archivo.name)


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 70)
print("PASO 2 FINALIZADO")
print("=" * 70)

print("Dataset cargado correctamente.")
print("Registros verificados:", len(datos_predictivos))
print("Columnas verificadas:", len(datos_predictivos.columns))
print("Variable objetivo: no_asistio")
print("Evidencias guardadas en:", carpeta_fuentes)
print("Modelos entrenados: 0")
print("SQL Server: SIN MODIFICACIONES")


RED SALUD INTEGRAL CARAZO
PASO 2. VERIFICACIÓN DEL DATASET PREDICTIVO

1. DIMENSIONES DEL DATASET
Registros: 8996
Columnas: 13

2. DICCIONARIO DE COLUMNAS


,Columna,Tipo_dato,Valores_nulos,Valores_distintos
0,id_cita,str,0,8996
1,fecha_cita,object,0,181
2,anio,int64,0,1
3,mes,int64,0,6
4,centro_salud,str,0,4
5,especialidad,str,0,5
6,canal_agendamiento,str,0,4
7,grupo_edad,str,0,4
8,tipo_cobertura,str,0,3
9,espera_dias,int64,0,61



3. VALORES FALTANTES


,Columna,Nulos,Porcentaje_nulos
0,id_cita,0,0.0
1,fecha_cita,0,0.0
2,anio,0,0.0
3,mes,0,0.0
4,centro_salud,0,0.0
5,especialidad,0,0.0
6,canal_agendamiento,0,0.0
7,grupo_edad,0,0.0
8,tipo_cobertura,0,0.0
9,espera_dias,0,0.0


Total de valores nulos: 0

4. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO


...


5. VERIFICACIÓN DE IDENTIFICADORES
Identificadores duplicados: 0
Identificadores nulos: 0

6. VERIFICACIÓN DE FECHAS
Fechas inválidas: 0
Fecha mínima: 2026-01-01 00:00:00
Fecha máxima: 2026-06-30 00:00:00

7. DISTRIBUCIÓN MENSUAL


,Periodo,Citas,Inasistencias,Asistencias,Tasa_inasistencia_pct
0,2026-01,1539,597,942,38.79
1,2026-02,1362,510,852,37.44
2,2026-03,1519,582,937,38.31
3,2026-04,1528,569,959,37.24
4,2026-05,1511,560,951,37.06
5,2026-06,1537,563,974,36.63



8. PRIMEROS CINCO REGISTROS


,id_cita,fecha_cita,anio,mes,centro_salud,especialidad,canal_agendamiento,grupo_edad,tipo_cobertura,espera_dias,citas_previas_perdidas,recordatorios_enviados,no_asistio
0,RSV2-000001,2026-04-09,2026,4,DIRIAMBA,MEDICINA GENERAL,WEB,60+,MIXTA,33,1,3,0
1,RSV2-000002,2026-04-30,2026,4,JINOTEPE,LABORATORIO,VENTANILLA,45-59,PRIVADA,6,4,3,1
2,RSV2-000004,2026-03-28,2026,3,DIRIAMBA,GINECOLOGÍA,WEB,30-44,PRIVADA,11,0,1,0
3,RSV2-000006,2026-01-06,2026,1,JINOTEPE,LABORATORIO,CALL CENTER,60+,PÚBLICA,35,4,2,1
4,RSV2-000007,2026-04-27,2026,4,SAN MARCOS,LABORATORIO,VENTANILLA,30-44,PÚBLICA,12,1,0,0



ARCHIVOS GUARDADOS CORRECTAMENTE
Ubicación: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados\01_fuentes
- diccionario_columnas.csv
- diccionario_datos.csv
- distribucion_mensual.csv
- distribucion_objetivo.csv
- valores_faltantes.csv

PASO 2 FINALIZADO
Dataset cargado correctamente.
Registros verificados: 8996
Columnas verificadas: 13
Variable objetivo: no_asistio
Evidencias guardadas en: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados\01_fuentes
Modelos entrenados: 0
SQL Server: SIN MODIFICACIONES



## PASO 3. Preparación de variables predictoras

### Objetivo
Seleccionar y preparar las características del dataset que servirán para entrenar modelos predictivos de inasistencia a citas médicas.

### Variables numéricas
- espera_dias
- citas_previas_perdidas

### Variables categóricas
- centro_salud
- especialidad
- canal_agendamiento
- grupo_edad
- tipo_cobertura
- mes

### Variable objetivo
no_asistio:
- 0: Asistió.
- 1: No asistió.

### Criterios de preparación
- Validar los tipos de datos.
- Comprobar valores faltantes o inválidos.
- Verificar la consistencia de año y mes.
- Excluir identificadores de las variables predictoras.
- Conservar las evidencias para su posterior auditoría.

### Consideración metodológica
La variable citas_previas_perdidas se incorpora a un experimento exploratorio para contrastar el rendimiento histórico. Su disponibilidad temporal deberá verificarse antes de considerar definitivos los modelos.

Las transformaciones de aprendizaje automático se ajustarán posteriormente únicamente sobre los registros de entrenamiento.


In [4]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# MINERÍA DE DATOS
# PASO 3. PREPARACIÓN DE VARIABLES PREDICTORAS
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. PREPARAR EL ENTORNO
# ============================================================

proyecto = Path.cwd()

if "datos_predictivos" not in globals():
    raise RuntimeError(
        "Primero debes ejecutar el PASO 2."
    )

df_modelo = datos_predictivos.copy()

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PASO 3. PREPARACIÓN DE VARIABLES")
print("=" * 70)

print("\nRegistros iniciales:", len(df_modelo))
print("Columnas iniciales:", len(df_modelo.columns))


# ============================================================
# 2. DEFINIR VARIABLE OBJETIVO
# ============================================================

OBJETIVO = "no_asistio"


# ============================================================
# 3. DEFINIR VARIABLES PREDICTORAS
# ============================================================

# Variables numéricas
NUM = [
    "espera_dias",
    "citas_previas_perdidas"
]

# Variables categóricas
CAT = [
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "tipo_cobertura",
    "mes"
]

# Lista de todas las variables predictoras
FEATURES = NUM + CAT

# Variables auxiliares de auditoría
IDENTIFICADOR = "id_cita"
FECHA = "fecha_cita"


# ============================================================
# 4. VERIFICAR COLUMNAS OBLIGATORIAS
# ============================================================

columnas_requeridas = (
    FEATURES
    + [OBJETIVO, IDENTIFICADOR, FECHA, "anio"]
)

faltantes = set(columnas_requeridas) - set(
    df_modelo.columns
)

if faltantes:
    raise ValueError(
        f"Columnas faltantes: {sorted(faltantes)}"
    )

print("\nColumnas obligatorias: VERIFICADAS")


# ============================================================
# 5. PREPARAR Y VERIFICAR FECHAS
# ============================================================

df_modelo[FECHA] = pd.to_datetime(
    df_modelo[FECHA],
    errors="coerce"
)

fechas_invalidas = int(
    df_modelo[FECHA].isna().sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        f"Fechas inválidas: {fechas_invalidas}"
    )

# Comprobar el año y mes registrados en SQL Server
anios_originales = pd.to_numeric(
    df_modelo["anio"],
    errors="coerce"
)

meses_originales = pd.to_numeric(
    df_modelo["mes"],
    errors="coerce"
)

errores_anio = int(
    (
        anios_originales
        != df_modelo[FECHA].dt.year
    ).sum()
)

errores_mes = int(
    (
        meses_originales
        != df_modelo[FECHA].dt.month
    ).sum()
)

if errores_anio > 0 or errores_mes > 0:
    raise ValueError(
        "Existen inconsistencias entre fecha_cita, "
        "anio y mes."
    )

print("\nFechas válidas:", len(df_modelo))
print("Inconsistencias de año:", errores_anio)
print("Inconsistencias de mes:", errores_mes)


# ============================================================
# 6. PREPARAR VARIABLE OBJETIVO
# ============================================================

df_modelo[OBJETIVO] = pd.to_numeric(
    df_modelo[OBJETIVO],
    errors="coerce"
)

if not df_modelo[OBJETIVO].isin([0, 1]).all():
    raise ValueError(
        "La variable objetivo contiene valores inválidos."
    )

df_modelo[OBJETIVO] = (
    df_modelo[OBJETIVO].astype(int)
)


# ============================================================
# 7. PREPARAR VARIABLES NUMÉRICAS
# ============================================================

for columna in NUM:

    df_modelo[columna] = pd.to_numeric(
        df_modelo[columna],
        errors="coerce"
    )

    nulos = df_modelo[columna].isna().sum()

    negativos = (
        df_modelo[columna] < 0
    ).sum()

    decimales = (
        df_modelo[columna].notna()
        & (
            df_modelo[columna]
            != df_modelo[columna].round()
        )
    ).sum()

    if nulos or negativos or decimales:

        raise ValueError(
            f"Variable {columna}: "
            f"nulos={nulos}, "
            f"negativos={negativos}, "
            f"no_enteros={decimales}"
        )

print("\nVariables numéricas: VERIFICADAS")


# ============================================================
# 8. PREPARAR VARIABLES CATEGÓRICAS
# ============================================================

for columna in CAT:

    if columna == "mes":

        # Representar el mes como categoría
        # y no como valor numérico continuo.

        df_modelo[columna] = (
            df_modelo[FECHA]
            .dt.month
            .astype(str)
        )

    else:

        df_modelo[columna] = (
            df_modelo[columna]
            .astype("string")
            .str.strip()
        )

        if (
            df_modelo[columna].isna().any()
            or df_modelo[columna].eq("").any()
        ):
            raise ValueError(
                f"Existen categorías vacías en {columna}."
            )

        # Conservar valores como texto estándar
        df_modelo[columna] = (
            df_modelo[columna].astype(str)
        )

print("Variables categóricas: VERIFICADAS")


# ============================================================
# 9. VERIFICAR IDENTIFICADORES
# ============================================================

if (
    df_modelo[IDENTIFICADOR].isna().any()
    or df_modelo[IDENTIFICADOR].duplicated().any()
):
    raise ValueError(
        "Existen identificadores nulos o duplicados."
    )


# ============================================================
# 10. DEFINIR DATASET PARA MODELADO
# ============================================================

# Conservar ID y fecha para auditoría.
# NO serán variables predictoras.

columnas_modelado = [
    IDENTIFICADOR,
    FECHA
] + FEATURES + [OBJETIVO]

dataset_modelado = (
    df_modelo[columnas_modelado]
    .copy()
)

# Preparar conjunto de predictores y objetivo.
# Todavía NO se entrenarán modelos.

X = dataset_modelado[FEATURES].copy()
y = dataset_modelado[OBJETIVO].copy()

assert OBJETIVO not in X.columns
assert IDENTIFICADOR not in X.columns
assert FECHA not in X.columns

assert len(X) == len(y)
assert len(X) == len(df_modelo)


# ============================================================
# 11. CONSTRUIR DICCIONARIO DE VARIABLES
# ============================================================

resumen_variables = []

for columna in FEATURES:

    tipo_variable = (
        "NUMERICA"
        if columna in NUM
        else "CATEGORICA"
    )

    estado_temporal = (
        "PENDIENTE_VERIFICACION"
        if columna == "citas_previas_perdidas"
        else "REVISAR_DISPONIBILIDAD_PREVIA"
    )

    resumen_variables.append({
        "Variable": columna,
        "Tipo": tipo_variable,
        "Valores_distintos": int(
            dataset_modelado[columna].nunique()
        ),
        "Valores_faltantes": int(
            dataset_modelado[columna].isna().sum()
        ),
        "Estado_temporal": estado_temporal
    })

tabla_variables = pd.DataFrame(
    resumen_variables
)


# ============================================================
# 12. ESTADÍSTICAS NUMÉRICAS
# ============================================================

print("\n" + "=" * 70)
print("1. VARIABLES PREDICTORAS")
print("=" * 70)

display(tabla_variables)

print("\n2. ESTADÍSTICAS NUMÉRICAS")

display(
    dataset_modelado[NUM]
    .describe()
    .round(2)
)


# ============================================================
# 13. DISTRIBUCIÓN DEL OBJETIVO
# ============================================================

tabla_objetivo = (
    dataset_modelado[OBJETIVO]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("no_asistio")
    .reset_index(name="Cantidad")
)

tabla_objetivo["Porcentaje"] = (
    tabla_objetivo["Cantidad"]
    / len(dataset_modelado)
    * 100
).round(2)

print("\n3. DISTRIBUCIÓN DEL OBJETIVO")
display(tabla_objetivo)


# ============================================================
# 14. DISTRIBUCIÓN DE CITAS PREVIAS PERDIDAS
# ============================================================

tabla_citas_previas = (
    dataset_modelado
    .groupby("citas_previas_perdidas")
    .agg(
        Total_citas=(OBJETIVO, "size"),
        Inasistencias=(OBJETIVO, "sum")
    )
    .reset_index()
)

tabla_citas_previas["Tasa_inasistencia_pct"] = (
    tabla_citas_previas["Inasistencias"]
    / tabla_citas_previas["Total_citas"]
    * 100
).round(2)

print("\n4. INASISTENCIA SEGÚN CITAS PREVIAS")
display(tabla_citas_previas)


# ============================================================
# 15. VERIFICAR DIMENSIONES
# ============================================================

print("\n" + "=" * 70)
print("5. DATASET PREPARADO")
print("=" * 70)

print("Registros:", len(dataset_modelado))
print("Variables predictoras:", len(FEATURES))
print("Variables numéricas:", len(NUM))
print("Variables categóricas:", len(CAT))
print("Variable objetivo:", OBJETIVO)

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)


# ============================================================
# 16. GUARDAR EVIDENCIAS
# ============================================================

carpeta_preparacion = (
    proyecto
    / "resultados"
    / "03_preparacion"
)

carpeta_preparacion.mkdir(
    parents=True,
    exist_ok=True
)

tabla_variables.to_csv(
    carpeta_preparacion
    / "diccionario_variables_modelado.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_citas_previas.to_csv(
    carpeta_preparacion
    / "analisis_citas_previas.csv",
    index=False,
    encoding="utf-8-sig"
)

dataset_modelado.to_csv(
    carpeta_preparacion
    / "dataset_preparado.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 70)
print("PASO 3 FINALIZADO")
print("=" * 70)

print("Dataset preparado correctamente.")
print("Variables predictoras definidas:", len(FEATURES))
print("Objetivo separado correctamente.")
print("Archivos guardados en:", carpeta_preparacion)
print("Modelos entrenados: 0")
print("SQL Server: SIN MODIFICACIONES")

print(
    "\nADVERTENCIA: citas_previas_perdidas "
    "sigue pendiente de verificación temporal."
)


RED SALUD INTEGRAL CARAZO
PASO 3. PREPARACIÓN DE VARIABLES

Registros iniciales: 8996
Columnas iniciales: 13

Columnas obligatorias: VERIFICADAS

Fechas válidas: 8996
Inconsistencias de año: 0
Inconsistencias de mes: 0

Variables numéricas: VERIFICADAS
Variables categóricas: VERIFICADAS

1. VARIABLES PREDICTORAS


,Variable,Tipo,Valores_distintos,Valores_faltantes,Estado_temporal
0,espera_dias,NUMERICA,61,0,REVISAR_DISPONIBILIDAD_PREVIA
1,citas_previas_perdidas,NUMERICA,5,0,PENDIENTE_VERIFICACION
2,centro_salud,CATEGORICA,4,0,REVISAR_DISPONIBILIDAD_PREVIA
3,especialidad,CATEGORICA,5,0,REVISAR_DISPONIBILIDAD_PREVIA
4,canal_agendamiento,CATEGORICA,4,0,REVISAR_DISPONIBILIDAD_PREVIA
5,grupo_edad,CATEGORICA,4,0,REVISAR_DISPONIBILIDAD_PREVIA
6,tipo_cobertura,CATEGORICA,3,0,REVISAR_DISPONIBILIDAD_PREVIA
7,mes,CATEGORICA,6,0,REVISAR_DISPONIBILIDAD_PREVIA



2. ESTADÍSTICAS NUMÉRICAS


,espera_dias,citas_previas_perdidas
count,8996.00,8996.00
mean,19.93,1.41
std,16.19,1.26
min,0.00,0.00
25%,7.00,0.00
50%,15.00,1.00
75%,29.00,2.00
max,60.00,4.00



3. DISTRIBUCIÓN DEL OBJETIVO


,no_asistio,Cantidad,Porcentaje
0,0,5615,62.42
1,1,3381,37.58



4. INASISTENCIA SEGÚN CITAS PREVIAS


,citas_previas_perdidas,Total_citas,Inasistencias,Tasa_inasistencia_pct
0,0,2809,232,8.26
1,1,2282,425,18.62
2,2,2013,1117,55.49
3,3,1193,966,80.97
4,4,699,641,91.70



5. DATASET PREPARADO
Registros: 8996
Variables predictoras: 8
Variables numéricas: 2
Variables categóricas: 6
Variable objetivo: no_asistio

Dimensiones de X: (8996, 8)
Dimensiones de y: (8996,)

PASO 3 FINALIZADO
Dataset preparado correctamente.
Variables predictoras definidas: 8
Objetivo separado correctamente.
Archivos guardados en: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados\03_preparacion
Modelos entrenados: 0
SQL Server: SIN MODIFICACIONES

ADVERTENCIA: citas_previas_perdidas sigue pendiente de verificación temporal.



## PASO 4. División cronológica del dataset

### Objetivo
Separar los registros predictivos en tres conjuntos independientes para entrenar, validar y evaluar los modelos de minería de datos.

### Distribución temporal
- Entrenamiento: enero a abril de 2026.
- Validación: mayo de 2026.
- Prueba final: junio de 2026.

### Criterios metodológicos
- Todos los algoritmos utilizarán la misma división.
- Los modelos se entrenarán exclusivamente con enero-abril.
- La comparación y selección de los mejores modelos se realizará con mayo.
- Junio se reservará para la evaluación final de los tres modelos seleccionados.
- Se verificará que no existan identificadores repetidos entre conjuntos.
- Las variables predictoras estarán separadas de la variable objetivo.

### Consideración
La variable citas_previas_perdidas permanece pendiente de verificación temporal antes de considerar definitivos los resultados predictivos.


In [5]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# MINERÍA DE DATOS
# PASO 4. DIVISIÓN CRONOLÓGICA DEL DATASET
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. VERIFICAR DATOS DEL PASO 3
# ============================================================

if "dataset_modelado" not in globals():
    raise RuntimeError(
        "Debes ejecutar primero el PASO 3."
    )

if "FEATURES" not in globals():
    raise RuntimeError(
        "No se encontraron las variables predictoras."
    )

if "OBJETIVO" not in globals():
    raise RuntimeError(
        "No se encontró la variable objetivo."
    )

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PASO 4. DIVISIÓN CRONOLÓGICA")
print("=" * 70)


# ============================================================
# 2. PREPARAR COPIA DE LOS DATOS
# ============================================================

df_particion = dataset_modelado.copy()

df_particion["fecha_cita"] = pd.to_datetime(
    df_particion["fecha_cita"],
    errors="coerce"
)

if df_particion["fecha_cita"].isna().any():
    raise ValueError(
        "Existen fechas inválidas."
    )

if df_particion["id_cita"].duplicated().any():
    raise ValueError(
        "Existen identificadores duplicados."
    )

if not df_particion[OBJETIVO].isin([0, 1]).all():
    raise ValueError(
        "La variable objetivo contiene valores inválidos."
    )


# ============================================================
# 3. DIVISIÓN CRONOLÓGICA
# ============================================================

# Entrenamiento: enero a abril
train = df_particion.loc[
    (df_particion["fecha_cita"] >= "2026-01-01")
    & (df_particion["fecha_cita"] < "2026-05-01")
].copy()

# Validación: mayo
val = df_particion.loc[
    (df_particion["fecha_cita"] >= "2026-05-01")
    & (df_particion["fecha_cita"] < "2026-06-01")
].copy()

# Prueba final: junio
test = df_particion.loc[
    (df_particion["fecha_cita"] >= "2026-06-01")
    & (df_particion["fecha_cita"] < "2026-07-01")
].copy()


# ============================================================
# 4. VERIFICAR LAS PARTICIONES
# ============================================================

conjuntos = {
    "ENTRENAMIENTO": train,
    "VALIDACION": val,
    "PRUEBA": test
}

for nombre, datos in conjuntos.items():

    if datos.empty:
        raise ValueError(
            f"El conjunto {nombre} está vacío."
        )

    if datos[OBJETIVO].nunique() != 2:
        raise ValueError(
            f"{nombre} no contiene ambas clases."
        )

total_particiones = (
    len(train) + len(val) + len(test)
)

if total_particiones != len(df_particion):
    raise ValueError(
        "Existen registros fuera de los períodos."
    )

# Comprobar separación temporal
assert (
    train["fecha_cita"].max()
    < val["fecha_cita"].min()
)

assert (
    val["fecha_cita"].max()
    < test["fecha_cita"].min()
)

# Comprobar que no se repiten citas
ids_train = set(train["id_cita"])
ids_val = set(val["id_cita"])
ids_test = set(test["id_cita"])

assert ids_train.isdisjoint(ids_val)
assert ids_train.isdisjoint(ids_test)
assert ids_val.isdisjoint(ids_test)


# ============================================================
# 5. SEPARAR PREDICTORAS Y OBJETIVO
# ============================================================

X_train = train[FEATURES].copy()
y_train = train[OBJETIVO].astype(int).copy()

X_val = val[FEATURES].copy()
y_val = val[OBJETIVO].astype(int).copy()

# La prueba final permanecerá reservada.
X_test = test[FEATURES].copy()
y_test = test[OBJETIVO].astype(int).copy()

# Verificar que no hay columnas excluidas
assert OBJETIVO not in X_train.columns
assert "id_cita" not in X_train.columns
assert "fecha_cita" not in X_train.columns

assert list(X_train.columns) == list(X_val.columns)
assert list(X_val.columns) == list(X_test.columns)


# ============================================================
# 6. CONSTRUIR TABLA DE DISTRIBUCIÓN
# ============================================================

resumen = []

for nombre, datos in conjuntos.items():

    total = len(datos)

    inasistencias = int(
        datos[OBJETIVO].sum()
    )

    asistencias = total - inasistencias

    resumen.append({
        "Conjunto": nombre,
        "Registros": total,
        "Asistencias": asistencias,
        "Inasistencias": inasistencias,
        "Tasa_inasistencia_pct": round(
            inasistencias / total * 100,
            2
        ),
        "Fecha_minima": str(
            datos["fecha_cita"].min().date()
        ),
        "Fecha_maxima": str(
            datos["fecha_cita"].max().date()
        )
    })

tabla_particiones = pd.DataFrame(resumen)


# ============================================================
# 7. MOSTRAR RESULTADOS
# ============================================================

print("\n1. DISTRIBUCIÓN DE LOS CONJUNTOS")
display(tabla_particiones)

print("\n2. DIMENSIONES DE ENTRENAMIENTO")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\n3. DIMENSIONES DE VALIDACIÓN")

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\n4. DIMENSIONES DE PRUEBA FINAL")

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\n5. CONTROLES")

print("Variables predictoras:", len(FEATURES))
print("Variable objetivo:", OBJETIVO)
print("Total de registros:", total_particiones)
print("Identificadores compartidos: 0")
print("Separación temporal: VERIFICADA")


# ============================================================
# 8. GUARDAR EVIDENCIAS
# ============================================================

proyecto = Path.cwd()

carpeta_particion = (
    proyecto
    / "resultados"
    / "03_preparacion"
)

carpeta_particion.mkdir(
    parents=True,
    exist_ok=True
)

tabla_particiones.to_csv(
    carpeta_particion
    / "division_cronologica.csv",
    index=False,
    encoding="utf-8-sig"
)

# Evidencia de qué registros pertenecen
# a cada conjunto, sin entrenar modelos.

auditoria_particiones = pd.concat(
    [
        train.assign(
            Conjunto="ENTRENAMIENTO"
        ),
        val.assign(
            Conjunto="VALIDACION"
        ),
        test.assign(
            Conjunto="PRUEBA"
        )
    ],
    ignore_index=True
)

auditoria_particiones[
    [
        "id_cita",
        "fecha_cita",
        "Conjunto",
        OBJETIVO
    ]
].to_csv(
    carpeta_particion
    / "auditoria_particiones.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 70)
print("PASO 4 FINALIZADO")
print("=" * 70)

print("División cronológica: CORRECTA")
print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Prueba final:", len(test))

print("\nArchivos guardados en:")
print(carpeta_particion)

print("\nModelos entrenados: 0")
print("Datos de junio evaluados por modelos: NO")
print("SQL Server: SIN MODIFICACIONES")


RED SALUD INTEGRAL CARAZO
PASO 4. DIVISIÓN CRONOLÓGICA

1. DISTRIBUCIÓN DE LOS CONJUNTOS


,Conjunto,Registros,Asistencias,Inasistencias,Tasa_inasistencia_pct,Fecha_minima,Fecha_maxima
0,ENTRENAMIENTO,5948,3690,2258,37.96,2026-01-01,2026-04-30
1,VALIDACION,1511,951,560,37.06,2026-05-01,2026-05-31
2,PRUEBA,1537,974,563,36.63,2026-06-01,2026-06-30



2. DIMENSIONES DE ENTRENAMIENTO
X_train: (5948, 8)
y_train: (5948,)

3. DIMENSIONES DE VALIDACIÓN
X_val: (1511, 8)
y_val: (1511,)

4. DIMENSIONES DE PRUEBA FINAL
X_test: (1537, 8)
y_test: (1537,)

5. CONTROLES
Variables predictoras: 8
Variable objetivo: no_asistio
Total de registros: 8996
Identificadores compartidos: 0
Separación temporal: VERIFICADA

PASO 4 FINALIZADO
División cronológica: CORRECTA
Entrenamiento: 5948
Validación: 1511
Prueba final: 1537

Archivos guardados en:
c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados\03_preparacion

Modelos entrenados: 0
Datos de junio evaluados por modelos: NO
SQL Server: SIN MODIFICACIONES



## PASO 5. Configuración del preprocesamiento y los algoritmos

### Objetivo
Configurar siete algoritmos de aprendizaje automático para comparar su capacidad de predecir la inasistencia a citas médicas.

### Algoritmos seleccionados
1. Regresión Logística.
2. Árbol de Decisión.
3. Random Forest.
4. Gradient Boosting.
5. KNN (k=10).
6. Naive Bayes.
7. SVM (RBF).

### Preprocesamiento
- Imputación por mediana para las variables numéricas.
- Estandarización numérica para los algoritmos que la requieren.
- Imputación por moda para las variables categóricas.
- Codificación One-Hot para las categorías.
- Uso de Pipeline para prevenir que el preprocesamiento aprenda información de validación o prueba.

### Evaluación posterior
La comparación se realizará sobre mayo de 2026 utilizando Accuracy, Precision, Recall, F1, F2, ROC-AUC y PR-AUC.

Se seleccionarán tres modelos diferentes por su rendimiento de validación.

### Indicadores del proyecto
- KPI-M01: ROC-AUC.
- KPI-M02: Recall.
- KPI-C01: Completitud de los registros predictivos.

### Control metodológico
La procedencia temporal de citas_previas_perdidas deberá verificarse antes de considerar válidos los resultados predictivos finales.


In [6]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# MINERÍA DE DATOS
# PASO 5. CONFIGURACIÓN DE SIETE MODELOS
# ============================================================

import sys
import pandas as pd
import sklearn

from pathlib import Path
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC


# ============================================================
# 1. VERIFICAR LOS DATOS PREPARADOS
# ============================================================

variables_necesarias = [
    "NUM",
    "CAT",
    "FEATURES",
    "X_train",
    "y_train",
    "X_val",
    "y_val"
]

faltantes = [
    nombre
    for nombre in variables_necesarias
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecuta primero los PASOS 3 y 4. "
        f"Faltan: {faltantes}"
    )

assert len(X_train) == 5948
assert len(X_val) == 1511

assert len(FEATURES) == 8
assert len(NUM) == 2
assert len(CAT) == 6

assert list(X_train.columns) == FEATURES
assert list(X_val.columns) == FEATURES

assert "no_asistio" not in X_train.columns
assert "id_cita" not in X_train.columns

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PASO 5. CONFIGURACIÓN DE SIETE MODELOS")
print("=" * 70)

print("\nVersión de Python:", sys.version.split()[0])
print("Versión de scikit-learn:", sklearn.__version__)

print("\nEntrenamiento:", len(X_train))
print("Validación:", len(X_val))
print("Variables predictoras:", len(FEATURES))


# ============================================================
# 2. CREAR FUNCIÓN DE PREPROCESAMIENTO
# ============================================================

def crear_preprocesador(escalar=False):

    # Preparar variables numéricas
    pasos_numericos = [
        (
            "imputador",
            SimpleImputer(strategy="median")
        )
    ]

    if escalar:

        pasos_numericos.append(
            (
                "escalador",
                StandardScaler()
            )
        )

    preprocesamiento_numerico = Pipeline(
        pasos_numericos
    )

    # Compatibilidad con distintas versiones
    # de scikit-learn
    try:

        codificador = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

    except TypeError:

        codificador = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    preprocesamiento_categorico = Pipeline([
        (
            "imputador",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "codificador",
            codificador
        )
    ])

    # Integrar preprocesamiento
    preprocesador = ColumnTransformer(
        transformers=[
            (
                "numericas",
                preprocesamiento_numerico,
                NUM
            ),
            (
                "categoricas",
                preprocesamiento_categorico,
                CAT
            )
        ],
        remainder="drop"
    )

    return preprocesador


# ============================================================
# 3. CREAR FUNCIÓN PARA CONSTRUIR MODELOS
# ============================================================

def crear_pipeline(clasificador, escalar=False):

    return Pipeline([
        (
            "preprocesamiento",
            crear_preprocesador(
                escalar=escalar
            )
        ),
        (
            "clasificador",
            clasificador
        )
    ])


# ============================================================
# 4. CONFIGURAR LOS SIETE MODELOS
# ============================================================

modelos = {

    # MODELO 1. REGRESIÓN LOGÍSTICA
    "Regresión Logística": crear_pipeline(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ),
        escalar=True
    ),

    # MODELO 2. ÁRBOL DE DECISIÓN
    "Árbol de Decisión": crear_pipeline(
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_split=20,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42
        ),
        escalar=False
    ),

    # MODELO 3. RANDOM FOREST
    "Random Forest": crear_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ),
        escalar=False
    ),

    # MODELO 4. GRADIENT BOOSTING
    "Gradient Boosting": crear_pipeline(
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),
        escalar=False
    ),

    # MODELO 5. KNN
    "KNN (k=10)": crear_pipeline(
        KNeighborsClassifier(
            n_neighbors=10,
            weights="distance",
            n_jobs=-1
        ),
        escalar=True
    ),

    # MODELO 6. NAIVE BAYES
    "Naive Bayes": crear_pipeline(
        GaussianNB(),
        escalar=False
    ),

    # MODELO 7. SVM
    "SVM (RBF)": crear_pipeline(
        SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        ),
        escalar=True
    )
}


# ============================================================
# 5. DEFINIR MÉTRICAS DE COMPARACIÓN
# ============================================================

metricas_comparacion = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)",
    "F2-Score (%)",
    "ROC-AUC",
    "PR-AUC",
    "Tiempo (s)"
]

UMBRAL = 0.50


# ============================================================
# 6. CREAR TABLA DE CONFIGURACIÓN
# ============================================================

descripcion_modelos = {
    "Regresión Logística": "Clasificación lineal",
    "Árbol de Decisión": "Reglas de decisión",
    "Random Forest": "Conjunto de árboles",
    "Gradient Boosting": "Árboles secuenciales",
    "KNN (k=10)": "Vecinos más cercanos",
    "Naive Bayes": "Clasificador probabilístico",
    "SVM (RBF)": "Máquina de vectores de soporte"
}

configuracion = []

for numero, (nombre, pipeline) in enumerate(
    modelos.items(),
    start=1
):

    clasificador = pipeline.named_steps[
        "clasificador"
    ]

    parametros = clasificador.get_params()

    usa_escalado = nombre in [
        "Regresión Logística",
        "KNN (k=10)",
        "SVM (RBF)"
    ]

    configuracion.append({
        "Numero": numero,
        "Modelo": nombre,
        "Descripcion": descripcion_modelos[nombre],
        "Escalado": "SI" if usa_escalado else "NO",
        "Parametros": str(parametros)
    })

tabla_configuracion = pd.DataFrame(
    configuracion
)


# ============================================================
# 7. MOSTRAR RESULTADOS
# ============================================================

print("\n1. MODELOS CONFIGURADOS")

display(
    tabla_configuracion[
        [
            "Numero",
            "Modelo",
            "Descripcion",
            "Escalado"
        ]
    ]
)

print("\n2. VARIABLES PREDICTORAS")

display(
    pd.DataFrame({
        "Variable": FEATURES,
        "Tipo": [
            "NUMERICA"
            if variable in NUM
            else "CATEGORICA"
            for variable in FEATURES
        ]
    })
)

print("\n3. MÉTRICAS QUE UTILIZAREMOS")

for metrica in metricas_comparacion:
    print("-", metrica)

print("\n4. CONFIGURACIÓN GENERAL")

print("Modelos configurados:", len(modelos))
print("Registros de entrenamiento:", len(X_train))
print("Registros de validación:", len(X_val))
print("Umbral de clasificación:", UMBRAL)

print("Transformaciones ajustadas:", "NO")
print("Modelos entrenados:", 0)


# ============================================================
# 8. VERIFICAR LOS SIETE ALGORITMOS
# ============================================================

assert len(modelos) == 7

assert len(set(modelos.keys())) == 7

for nombre, modelo in modelos.items():

    assert isinstance(modelo, Pipeline)

    assert "preprocesamiento" in modelo.named_steps

    assert "clasificador" in modelo.named_steps

print("\nVERIFICACIÓN DE MODELOS: CORRECTA")


# ============================================================
# 9. GUARDAR CONFIGURACIÓN
# ============================================================

proyecto = Path.cwd()

carpeta_modelos = (
    proyecto
    / "resultados"
    / "04_modelos"
)

carpeta_modelos.mkdir(
    parents=True,
    exist_ok=True
)

tabla_configuracion.to_csv(
    carpeta_modelos
    / "configuracion_siete_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("PASO 5 FINALIZADO")
print("=" * 70)

print("Algoritmos configurados: 7")
print("Entrenamiento realizado: NO")
print("Validación realizada: NO")
print("Evaluación de junio realizada: NO")
print("SQL Server: SIN MODIFICACIONES")


RED SALUD INTEGRAL CARAZO
PASO 5. CONFIGURACIÓN DE SIETE MODELOS

Versión de Python: 3.14.3
Versión de scikit-learn: 1.9.1

Entrenamiento: 5948
Validación: 1511
Variables predictoras: 8

1. MODELOS CONFIGURADOS


,Numero,Modelo,Descripcion,Escalado
0,1,Regresión Logística,Clasificación lineal,SI
1,2,Árbol de Decisión,Reglas de decisión,NO
2,3,Random Forest,Conjunto de árboles,NO
3,4,Gradient Boosting,Árboles secuenciales,NO
4,5,KNN (k=10),Vecinos más cercanos,SI
5,6,Naive Bayes,Clasificador probabilístico,NO
6,7,SVM (RBF),Máquina de vectores de soporte,SI



2. VARIABLES PREDICTORAS


,Variable,Tipo
0,espera_dias,NUMERICA
1,citas_previas_perdidas,NUMERICA
2,centro_salud,CATEGORICA
3,especialidad,CATEGORICA
4,canal_agendamiento,CATEGORICA
5,grupo_edad,CATEGORICA
6,tipo_cobertura,CATEGORICA
7,mes,CATEGORICA



3. MÉTRICAS QUE UTILIZAREMOS
- Accuracy (%)
- Precision (%)
- Recall (%)
- F1-Score (%)
- F2-Score (%)
- ROC-AUC
- PR-AUC
- Tiempo (s)

4. CONFIGURACIÓN GENERAL
Modelos configurados: 7
Registros de entrenamiento: 5948
Registros de validación: 1511
Umbral de clasificación: 0.5
Transformaciones ajustadas: NO
Modelos entrenados: 0

VERIFICACIÓN DE MODELOS: CORRECTA

PASO 5 FINALIZADO
Algoritmos configurados: 7
Entrenamiento realizado: NO
Validación realizada: NO
Evaluación de junio realizada: NO
SQL Server: SIN MODIFICACIONES
